In [1]:
!pip install transformers pandas numpy scikit-learn torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 687.8 kB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Load the datasets
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
submission_df = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")

print("Training data shape:", train_df.shape)

2025-10-16 18:14:07.932133: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760638448.112024      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760638448.159702      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Training data shape: (7613, 5)


In [3]:
# Choose a model name
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 160 # Max length for a tweet

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_data(df, include_targets=True):
    """
    Tokenizes the text in a DataFrame and returns PyTorch tensors.
    """
    # Use the tokenizer's batch encoding which is highly efficient
    encodings = tokenizer.batch_encode_plus(
        df.text.tolist(),
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encodings['input_ids']
    attention_mask = encodings['attention_mask']

    if include_targets:
        targets = torch.tensor(df.target.to_numpy(), dtype=torch.long)
        return input_ids, attention_mask, targets
    else:
        return input_ids, attention_mask

# Split the training data for validation
df_train, df_val = train_test_split(train_df, test_size=0.1, random_state=42)

# Prepare all datasets
train_inputs, train_masks, train_targets = tokenize_data(df_train)
val_inputs, val_masks, val_targets = tokenize_data(df_val)
test_inputs, test_masks = tokenize_data(test_df, include_targets=False)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [4]:
BATCH_SIZE = 16

# Create TensorDatasets
train_dataset = TensorDataset(train_inputs, train_masks, train_targets)
val_dataset = TensorDataset(val_inputs, val_masks, val_targets)
test_dataset = TensorDataset(test_inputs, test_masks)

# Create DataLoaders
train_data_loader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)
val_data_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_data_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [5]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

# --- Training Setup ---
EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_data_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# --- Training Loop ---
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)

    model.train()
    total_loss = 0

    for input_ids, attention_mask, targets in tqdm(train_data_loader):
        # Move batch to the device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        targets = targets.to(device)

        # Clear any previously calculated gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=targets
        )

        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_loss / len(train_data_loader)
    print(f'Training loss: {avg_train_loss}')

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3
----------


  0%|          | 0/429 [00:00<?, ?it/s]

Training loss: 0.43277740445517715
Epoch 2/3
----------


  0%|          | 0/429 [00:00<?, ?it/s]

Training loss: 0.3137047887022123
Epoch 3/3
----------


  0%|          | 0/429 [00:00<?, ?it/s]

Training loss: 0.2312743751883576


In [6]:
# --- Prediction Loop ---
model.eval()
predictions = []

with torch.no_grad():
    for input_ids, attention_mask in tqdm(test_data_loader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        _, preds = torch.max(outputs.logits, dim=1)
        predictions.extend(preds.cpu().numpy()) # Move predictions to CPU

  0%|          | 0/204 [00:00<?, ?it/s]

In [7]:
submission_df['target'] = predictions
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
submission_df.head()

Submission file created successfully!


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
